In [1]:
# ============================================================
# Institutional Freedom vs. Linguistic Complexity
# Composite complexity score from multiple WALS features:
#   22A - Inflectional Synthesis of the Verb
#   21B - Exponence of Tense-Aspect-Mood Categories
#   20A - Fusion of Selected Inflectional Formatives
#   26A - Prefixing vs. Suffixing in Inflectional Morphology
# ============================================================

# 1. INSTALL TOOLS
!pip install country_converter adjustText plotly --upgrade

import pandas as pd
import statsmodels.api as sm
import numpy as np
import country_converter as coco
import requests
from io import StringIO
import plotly.graph_objects as go


# ╔══════════════════════════════════════════════════════════╗
# ║                   CONFIGURATION                          ║
# ╠══════════════════════════════════════════════════════════╣
# ║  Toggle these two switches before running.               ║
# ╚══════════════════════════════════════════════════════════╝

EXCLUDE_PIVOT_LANGUAGES = False   # True = remove global trade/colonial languages
LOG_TRANSFORM_COMPLEXITY = False  # True = log-transform complexity score before OLS


# 2. FETCH & CLEAN HUMAN FREEDOM INDEX (HFI)
hfi_url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2022/2022-02-22/freedom.csv"
try:
    hfi_raw = pd.read_csv(hfi_url)
    hfi_raw.columns = hfi_raw.columns.str.lower()
    cc = coco.CountryConverter()
    hfi_raw['iso'] = cc.convert(names=hfi_raw['country'], to='ISO3')
    hfi_raw['freedom_score'] = 7 - ((hfi_raw['cl'] + hfi_raw['pr']) / 2)
    hfi_df = hfi_raw[hfi_raw['year'] == 2020][['iso', 'country', 'freedom_score']].copy()
    print(f"HFI loaded: {len(hfi_df)} countries")
except Exception as e:
    print(f"HFI Error: {e}")


# 3. FETCH MULTIPLE WALS FEATURES AND BUILD COMPOSITE SCORE
WALS_FEATURES = {
    "22A": "synthesis",
    "21B": "exponence",
    "20A": "fusion",
    "26A": "prefixsuffix",
}

def fetch_wals_feature(feature_id):
    url = f"https://wals.info/feature/{feature_id}.tab"
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    lines = resp.text.splitlines()
    header_idx = next(i for i, l in enumerate(lines) if 'value' in l and 'description' in l)
    df = pd.read_csv(StringIO(resp.text), sep="\t", skiprows=header_idx)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={"name": "language"})
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["value"])[["language", "value"]].copy()
    df = df.rename(columns={"value": f"val_{feature_id}"})
    print(f"  WALS {feature_id}: {len(df)} languages, "
          f"value range {df[f'val_{feature_id}'].min():.0f}–{df[f'val_{feature_id}'].max():.0f}")
    return df

print("Fetching WALS features...")
wals_frames = []
for fid in WALS_FEATURES:
    try:
        wals_frames.append(fetch_wals_feature(fid))
    except Exception as e:
        print(f"  Failed {fid}: {e}")

wals_merged = wals_frames[0]
for frame in wals_frames[1:]:
    wals_merged = pd.merge(wals_merged, frame, on="language", how="outer")

val_cols = [c for c in wals_merged.columns if c.startswith("val_")]
for col in val_cols:
    vmin, vmax = wals_merged[col].min(), wals_merged[col].max()
    wals_merged[col + "_norm"] = (wals_merged[col] - vmin) / (vmax - vmin) * 100

norm_cols = [c for c in wals_merged.columns if c.endswith("_norm")]
wals_merged["complexity_score"] = wals_merged[norm_cols].mean(axis=1)
wals_merged = wals_merged.dropna(subset=["complexity_score"])

print(f"\nComposite scores computed for {len(wals_merged)} languages")
print(f"Score range: {wals_merged['complexity_score'].min():.1f}–{wals_merged['complexity_score'].max():.1f}")


# 4. PIVOT LANGUAGE EXCLUSION LIST
# Each language is flagged for one of:
#   (a) >100M L2 speakers globally
#   (b) Colonial administrative spread across multiple continents
#   (c) Designated UN working language
#   (d) Standardised/simplified form imposed over diverse speech communities
PIVOT_LANGUAGES = {
    "English",          # (a)(b)(c) ~1.5B L2 speakers, UN language, colonial spread
    "French",           # (b)(c)    UN language, 29-country official presence
    "Spanish",          # (a)(b)    Colonial spread across Americas
    "Portuguese",       # (b)       Colonial spread across Americas, Africa, Asia
    "Arabic",           # (a)(c)    UN language; Modern Standard Arabic is a
                        #           codified prestige form over native dialects
    "Mandarin Chinese", # (d)       Administrative language imposed over ~700
                        #           distinct Chinese speech communities
    "Russian",          # (b)(d)    Soviet-era lingua franca across Central Asia
    "Malay",            # (a)(d)    Maritime SE Asia trade lingua franca
    "Indonesian",       # (d)       Standardised from Malay as national unifier
                        #           over ~700 indigenous languages
    "Swahili",          # (d)       East African trade lingua franca; simplified
                        #           Bantu morphology vs. inland relatives
}


# 5. MAP PRIMARY LANGUAGE → COUNTRY ISO3
primary_language = {
    # Europe
    "ALB": "Albanian",          "ARM": "Armenian",
    "AUT": "German",            "BEL": "Dutch",
    "BIH": "Bosnian",           "BGR": "Bulgarian",
    "HRV": "Croatian",          "CZE": "Czech",
    "DNK": "Danish",            "EST": "Estonian",
    "FIN": "Finnish",           "FRA": "French",
    "DEU": "German",            "GRC": "Greek (Modern)",
    "HUN": "Hungarian",         "ISL": "Icelandic",
    "IRL": "Irish",             "ITA": "Italian",
    "LVA": "Latvian",           "LTU": "Lithuanian",
    "LUX": "French",            "MKD": "Macedonian",
    "MDA": "Romanian",          "MNE": "Serbian",
    "NLD": "Dutch",             "NOR": "Norwegian",
    "POL": "Polish",            "PRT": "Portuguese",
    "ROU": "Romanian",          "RUS": "Russian",
    "SRB": "Serbian",           "SVK": "Slovak",
    "SVN": "Slovenian",         "ESP": "Spanish",
    "SWE": "Swedish",           "CHE": "German",
    "UKR": "Ukrainian",         "GBR": "English",
    "BLR": "Belarusian",        "KOS": "Albanian",
    # Americas
    "USA": "English",           "CAN": "English",
    "MEX": "Spanish",           "GTM": "Spanish",
    "BLZ": "English",           "HND": "Spanish",
    "SLV": "Spanish",           "NIC": "Spanish",
    "CRI": "Spanish",           "PAN": "Spanish",
    "CUB": "Spanish",           "DOM": "Spanish",
    "HTI": "Haitian Creole",    "JAM": "English",
    "COL": "Spanish",           "VEN": "Spanish",
    "GUY": "English",           "SUR": "Dutch",
    "ECU": "Spanish",           "PER": "Spanish",
    "BOL": "Spanish",           "BRA": "Portuguese",
    "PRY": "Spanish",           "URY": "Spanish",
    "ARG": "Spanish",           "CHL": "Spanish",
    "TTO": "English",
    # Africa
    "MAR": "Arabic",            "DZA": "Arabic",
    "TUN": "Arabic",            "LBY": "Arabic",
    "EGY": "Arabic",            "SDN": "Arabic",
    "SSD": "Arabic",            "MRT": "Arabic",
    "MLI": "Bambara",           "NER": "Hausa",
    "TCD": "Arabic",            "SEN": "Wolof",
    "GMB": "Wolof",             "GNB": "Portuguese",
    "GIN": "Fula",              "SLE": "English",
    "LBR": "English",           "CIV": "French",
    "GHA": "Twi",               "BFA": "Moore",
    "TGO": "Ewe",               "BEN": "Fon",
    "NGA": "Hausa",             "CMR": "French",
    "CAF": "Sango",             "GNQ": "Spanish",
    "GAB": "French",            "COG": "French",
    "COD": "Lingala",           "RWA": "Kinyarwanda",
    "BDI": "Kirundi",           "UGA": "Luganda",
    "KEN": "Swahili",           "TZA": "Swahili",
    "ETH": "Amharic",           "ERI": "Tigrinya",
    "SOM": "Somali",            "DJI": "Somali",
    "ZMB": "Bemba",             "MWI": "Chichewa",
    "MOZ": "Portuguese",        "ZWE": "Shona",
    "BWA": "Tswana",            "NAM": "English",
    "ZAF": "Zulu",              "LSO": "Sotho, Southern",
    "SWZ": "Swati",             "MDG": "Malagasy",
    "AGO": "Portuguese",        "STP": "Portuguese",
    # Middle East & Central Asia
    "TUR": "Turkish",           "SYR": "Arabic",
    "LBN": "Arabic",            "ISR": "Hebrew",
    "JOR": "Arabic",            "SAU": "Arabic",
    "YEM": "Arabic",            "OMN": "Arabic",
    "ARE": "Arabic",            "QAT": "Arabic",
    "KWT": "Arabic",            "BHR": "Arabic",
    "IRQ": "Arabic",            "IRN": "Persian",
    "AFG": "Pashto",            "KAZ": "Kazakh",
    "UZB": "Uzbek",             "TKM": "Turkmen",
    "KGZ": "Kyrgyz",            "TJK": "Tajik",
    "GEO": "Georgian",          "AZE": "Azerbaijani",
    # South & Southeast Asia
    "PAK": "Urdu",              "IND": "Hindi",
    "BGD": "Bengali",           "LKA": "Sinhala",
    "NPL": "Nepali",            "BTN": "Dzongkha",
    "MMR": "Burmese",           "THA": "Thai",
    "LAO": "Lao",               "VNM": "Vietnamese",
    "KHM": "Khmer",             "MYS": "Malay",
    "SGP": "Malay",             "IDN": "Indonesian",
    "PHL": "Tagalog",           "TLS": "Tetun",
    # East Asia & Pacific
    "CHN": "Mandarin Chinese",  "TWN": "Mandarin Chinese",
    "JPN": "Japanese",          "KOR": "Korean",
    "MNG": "Mongolian",         "PRK": "Korean",
    "AUS": "English",           "NZL": "English",
    "PNG": "Tok Pisin",         "FJI": "Fijian",
}

lang_lookup = wals_merged.set_index("language")["complexity_score"].to_dict()

lang_rows = []
excluded_countries = []
for iso, lang_name in primary_language.items():
    score = lang_lookup.get(lang_name)
    if score is None:
        continue
    if EXCLUDE_PIVOT_LANGUAGES and lang_name in PIVOT_LANGUAGES:
        excluded_countries.append(iso)
        continue
    lang_rows.append({"iso": iso, "language": lang_name, "complexity_score": score})

lang_df = pd.DataFrame(lang_rows)

if EXCLUDE_PIVOT_LANGUAGES:
    print(f"Pivot languages excluded — {len(excluded_countries)} countries removed")
print(f"Country–language pairs retained: {len(lang_df)}")


# 6. MERGE, TRANSFORM & REGRESSION
if 'hfi_df' in locals() and not lang_df.empty:
    merged_df = pd.merge(hfi_df, lang_df, on='iso', how='inner')
    print(f"Final merged dataset: {len(merged_df)} countries")

    if not merged_df.empty:

        # ── LOG TRANSFORM ────────────────────────────────────────────────
        # The raw complexity score is right-skewed (kurtosis ~15 in baseline
        # run). Log-transforming pulls in extreme high values and makes the
        # residual distribution closer to normal, satisfying OLS assumptions.
        # The y-axis will show log(complexity) but interpretation is the same:
        # higher = more complex. Coefficient now means: a 1-unit increase in
        # freedom_score is associated with a β% change in complexity.

        if LOG_TRANSFORM_COMPLEXITY:
            # Add small constant to avoid log(0) if any score == 0
            merged_df['y'] = np.log(merged_df['complexity_score'] + 1)
            y_axis_label = 'Log Morphological Complexity Score  (ln(score+1))'
        else:
            merged_df['y'] = merged_df['complexity_score']
            y_axis_label = 'Morphological Complexity Score  (composite, 0–100)'

        X     = sm.add_constant(merged_df['freedom_score'])
        model = sm.OLS(merged_df['y'], X).fit()

        # ── DETERMINE WHICH POINTS GET PERMANENT LABELS ──────────────────
        merged_df['predicted'] = model.predict(X)
        merged_df['residual']  = (merged_df['y'] - merged_df['predicted']).abs()

        ALWAYS_LABEL = {
            'USA', 'GBR', 'CHN', 'RUS', 'DEU', 'FRA', 'JPN', 'IND',
            'BRA', 'AUS', 'CAN', 'KOR', 'PRK', 'IRN', 'ISR', 'TUR',
            'ZAF', 'NGA', 'ETH', 'EGY', 'SAU', 'GEO',
        }
        outlier_isos = set(merged_df.nlargest(12, 'residual')['iso'])
        extreme_isos = (
            set(merged_df.nlargest(4,  'freedom_score')['iso']) |
            set(merged_df.nsmallest(4, 'freedom_score')['iso']) |
            set(merged_df.nlargest(4,  'y')['iso'])             |
            set(merged_df.nsmallest(4, 'y')['iso'])
        )
        label_isos = ALWAYS_LABEL | outlier_isos | extreme_isos

        # Only keep label_isos that are actually in the (possibly pivot-filtered) dataset
        label_isos = label_isos & set(merged_df['iso'])

        merged_df['show_label'] = merged_df['iso'].isin(label_isos)
        labeled_df   = merged_df[merged_df['show_label']]
        unlabeled_df = merged_df[~merged_df['show_label']]

        # ── REGRESSION LINE ───────────────────────────────────────────────
        x_range = np.linspace(merged_df['freedom_score'].min(),
                              merged_df['freedom_score'].max(), 200)
        y_range = model.params['const'] + model.params['freedom_score'] * x_range

        r2 = model.rsquared
        pv = model.pvalues.get('freedom_score', model.pvalues.iloc[1])

        # ── SUBTITLE reflects active config ──────────────────────────────
        pivot_note = "pivot languages excluded" if EXCLUDE_PIVOT_LANGUAGES else "all languages included"
        log_note   = "log-transformed" if LOG_TRANSFORM_COMPLEXITY else "untransformed"
        subtitle   = (
            f"WALS 22A+21B+20A+26A composite &nbsp;|&nbsp; {log_note} &nbsp;|&nbsp; "
            f"{pivot_note} &nbsp;|&nbsp; "
            f"R² = {r2:.3f} &nbsp; p = {pv:.3f} &nbsp; n = {len(merged_df)}"
        )

        # ── BUILD PLOTLY FIGURE ───────────────────────────────────────────
        fig = go.Figure()

        # Regression line
        fig.add_trace(go.Scatter(
            x=x_range, y=y_range,
            mode='lines',
            line=dict(color='#e74c3c', width=2.5),
            name='Regression line',
            hoverinfo='skip',
        ))

        # Unlabelled points — hover only
        fig.add_trace(go.Scatter(
            x=unlabeled_df['freedom_score'],
            y=unlabeled_df['y'],
            mode='markers',
            marker=dict(size=8, color='#5b9bd5', opacity=0.6,
                        line=dict(color='white', width=0.8)),
            name='Country (hover for name)',
            customdata=np.stack([
                unlabeled_df['country'],
                unlabeled_df['language'],
                unlabeled_df['freedom_score'].round(2),
                unlabeled_df['complexity_score'].round(1),
            ], axis=-1),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Primary language: %{customdata[1]}<br>"
                "Freedom score: %{customdata[2]}<br>"
                "Complexity score (raw): %{customdata[3]}<br>"
                "<extra></extra>"
            ),
        ))

        # Labelled points — permanent text + hover
        fig.add_trace(go.Scatter(
            x=labeled_df['freedom_score'],
            y=labeled_df['y'],
            mode='markers+text',
            marker=dict(size=9, color='#1a5fa8', opacity=0.85,
                        line=dict(color='white', width=0.8)),
            text=labeled_df['country'],
            textposition='top center',
            textfont=dict(size=9, color='#1a1a1a'),
            name='Notable country',
            customdata=np.stack([
                labeled_df['country'],
                labeled_df['language'],
                labeled_df['freedom_score'].round(2),
                labeled_df['complexity_score'].round(1),
            ], axis=-1),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Primary language: %{customdata[1]}<br>"
                "Freedom score: %{customdata[2]}<br>"
                "Complexity score (raw): %{customdata[3]}<br>"
                "<extra></extra>"
            ),
        ))

        fig.update_layout(
            title=dict(
                text=f"Institutional Freedom vs. Linguistic Complexity: Global Regression<br><sup>{subtitle}</sup>",
                font=dict(size=16),
                x=0.5, xanchor='center',
            ),
            xaxis=dict(
                title=dict(text='Freedom Score  (Higher = More Inclusive Institutions)', font=dict(size=13)),
                gridcolor='#eeeeee'
            ),
            yaxis=dict(
                title=dict(text=y_axis_label, font=dict(size=13)),
                gridcolor='#eeeeee'
            ),
            plot_bgcolor='white',
            paper_bgcolor='white',
            legend=dict(orientation='h', yanchor='bottom', y=1.01,
                        xanchor='right', x=1),
            width=1100, height=750,
            hoverlabel=dict(bgcolor='white', font_size=12),
        )

        fig.show()

        try:
            fig.write_image("freedom_vs_linguistic_complexity.png", scale=2)
            print("Static PNG saved.")
        except Exception:
            print("(pip install kaleido to enable static PNG export)")

        print("\n" + "="*60)
        print(model.summary())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 28.0 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
HFI loaded: 193 countries
Fetching WALS features...
  WALS 22A: 145 languages, value range 1–7
  WALS 21B: 160 languages, value range 1–6
  WALS 20A: 165 languages, value range 1–7
  WALS 26A: 969 languages, value range 1–6

Composite scores computed for 983 languages
Score range: 0.0–100.0
Country–language pairs retained: 109
Final merged dataset: 108 countries


(pip install kaleido to enable static PNG export)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.029
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     3.153
Date:                Mon, 09 Mar 2026   Prob (F-statistic):             0.0787
Time:                        21:49:08   Log-Likelihood:                -443.41
No. Observations:                 108   AIC:                             890.8
Df Residuals:                     106   BIC:                             896.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------